# Sector IDW vs nearest region on a synthetic open pit

Reproduces the experiment in Zelenskyi et al. (2020): build a pit surface from
scattered survey points and measure how the error changes with the grid step
and the number of survey points. The pit here is synthetic (`pitmap.synthetic`),
so the numbers will not match the paper, but the method and the comparison do.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pitmap import NearestRegion, SectorIDW
from pitmap.synthetic import grid, sample_survey_points, true_surface

## 1. Survey points

Random survey points over the domain, with 0.3 m measurement noise.

In [ ]:
xy, z = sample_survey_points(n=400, seed=0)
xy.shape, z.min().round(1), z.max().round(1)

## 2. Build the surface on a 30 m grid

The 2020 paper found 30 m to be the practical grid step. Both interpolators
share the same `fit` / `predict` interface. Sector IDW uses a 150 m search
radius, 8 sectors and power 2.

In [ ]:
ax_x, ax_y, nodes = grid(step=30.0)
z_true = true_surface(nodes[:, 0], nodes[:, 1])

models = {
    "Nearest region": NearestRegion(),
    "Sector IDW": SectorIDW(radius=150.0, power=2.0, n_sectors=8),
}
pred = {name: m.fit(xy, z).predict(nodes) for name, m in models.items()}


def rmse(a, b):
    # ignore grid nodes with no survey point within the search radius
    ok = ~np.isnan(a)
    return float(np.sqrt(np.mean((a[ok] - b[ok]) ** 2)))


{name: round(rmse(p, z_true), 2) for name, p in pred.items()}

## 3. Look at the surfaces

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
shape = (ax_y.size, ax_x.size)
for ax, (name, values) in zip(axes, [("True surface", z_true), *pred.items()]):
    im = ax.imshow(
        values.reshape(shape),
        extent=[ax_x[0], ax_x[-1], ax_y[0], ax_y[-1]],
        origin="lower",
        cmap="terrain",
    )
    ax.set_title(name)
    ax.set_aspect("equal")
fig.colorbar(im, ax=axes, label="Elevation, m")
plt.show()

## 4. Error vs grid step

Coarser grids smooth over the benches; finer grids cost more nodes for
little gain. Error is measured against the true surface at the grid nodes.

In [ ]:
steps = [10, 20, 30, 50, 75, 100]
rows = []
for step in steps:
    _, _, g = grid(step=float(step))
    zt = true_surface(g[:, 0], g[:, 1])
    rows.append(
        [step, g.shape[0]]
        + [rmse(m.fit(xy, z).predict(g), zt) for m in models.values()]
    )

print(f"{'step':>5} {'nodes':>7} " + " ".join(f"{n:>15}" for n in models))
for r in rows:
    print(f"{r[0]:>5} {r[1]:>7} " + " ".join(f"{v:>15.2f}" for v in r[2:]))

## 5. Error vs number of survey points

Grid step fixed at 30 m; survey density varies.

In [ ]:
counts = [100, 200, 400, 800, 1600]
errs = {name: [] for name in models}
for n in counts:
    xy_n, z_n = sample_survey_points(n=n, seed=1)
    for name, m in models.items():
        errs[name].append(rmse(m.fit(xy_n, z_n).predict(nodes), z_true))

plt.figure(figsize=(6, 4))
for name, e in errs.items():
    plt.plot(counts, e, marker="o", label=name)
plt.xscale("log")
plt.xlabel("Survey points")
plt.ylabel("RMSE, m")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## References

Zelenskyi, O. S., et al. (2020). Nauk. Visn. Nats. Hirn. Univ., 6, 121-126.
https://doi.org/10.33271/nvngu/2020-6/121